# Atlas recipe example

This notebook uses a tiny synthetic Atlas dataset and a discovered recipe to show the public Woodpecker recipe API with bundled plugin fixes.


In [ ]:
import woodpecker_atlas_plugin  # noqa: F401 - imports plugin fixes for editable installs

import woodpecker
from woodpecker.testing import make_atlas

Create a deterministic Atlas-like dataset with two issues: missing `project_id` metadata and overly strong compression settings.

In [ ]:
dataset = make_atlas(missing=["project_id"], seed=7)
dataset["pr"].encoding["complevel"] = 5

dataset

Load the discovered Atlas recipe and inspect its content.

The recipe is bundled in the Atlas plugin package and selects the bundled Atlas plugin fixes.


In [ ]:
recipe = woodpecker.recipe.get("atlas.basic")

recipe.model_dump()

In [ ]:
recipe.match.model_dump(), [step.id for step in recipe.steps]

In [ ]:
findings = woodpecker.recipe.check(dataset, recipe)
findings.fix_ids

A dry run reports what would change but leaves the dataset untouched.

In [ ]:
result = woodpecker.recipe.fix(dataset, recipe, dry_run=True)

result.stats, result.preview, dataset.attrs.get("project_id"), dataset["pr"].encoding["complevel"]

Apply the same recipe in memory with `dry_run=False`.

In [ ]:
write = woodpecker.recipe.fix(dataset, recipe, dry_run=False)

(
    write.stats,
    dataset.attrs["project_id"],
    dataset["pr"].encoding["complevel"],
    dataset["pr"].encoding["zlib"],
    dataset["pr"].encoding["shuffle"],
)

In [ ]:
recheck = woodpecker.recipe.check(dataset, recipe)
bool(recheck)